# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset citation:** Mugotitsa, B, Maina, D, Owoko, H, Akinyi, L, Greenfield, J, Todd, J, Kavu, T and Kiragga, A 2026. A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa. Frontiers.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Every reference uses the entity `@id` from the Croissant schema. Let's enumerate the record sets and their fields.

In [ ]:
# Display available record sets and their fields
record_sets = dataset.record_sets

print("Available record sets and fields:")
record_set_ids = []
field_mappings = {}
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    print("  Fields:")
    fields = rs.fields
    field_ids = []
    for f in fields:
        print(f"    - Field name: {f.name}")
        print(f"      @id: {f['@id']} (type: {getattr(f, 'data_type', 'unknown')})")
        field_ids.append(f['@id'])
    field_mappings[rs['@id']] = field_ids
    print("\n")

# Optionally show detailed records from the first available record set
if record_set_ids:
    print(f"Sample records from {record_set_ids[0]}:")
    for idx, x in enumerate(dataset.records(record_set=record_set_ids[0])):
        print(x)
        if idx == 2:
            break  # Show only first 3 records

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")
        print(f"Sample data for {record_set_id}:")
        display(df.head())
    else:
        print(f"No records found for record set {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Example:** Let's use the main survey table record set. We'll choose a numeric field (e.g., GAD-7 score), filter out low scores, normalize them, and group by Gender.

In [ ]:
# Identify the main survey record set (here, select the first record set with data)
main_record_set_id = None
for rid in record_set_ids:
    if rid in dataframes and not dataframes[rid].empty:
        main_record_set_id = rid
        break

df = dataframes[main_record_set_id]
columns = df.columns.tolist()

# Find numeric field candidates; here we look for fields ending with '_score' or '_total' likely representing GAD-7, PHQ-9, PSQ, etc.
numeric_field_candidates = [col for col in columns if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower() or '_score' in col.lower() or '_total' in col.lower()]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
else:
    # fallback: try first integer/float column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
print(f"Using numeric field: {numeric_field_id}")

# Choose a threshold and filter
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by a key attribute (e.g., Gender or Age Group)
group_candidates = [col for col in df.columns if 'gender' in col.lower() or 'sex' in col.lower() or 'age' in col.lower() or 'education' in col.lower()]
group_field = group_candidates[0] if group_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the chosen numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group (if available)
if group_field:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides rich, AI-ready survey data from Kilifi County, Kenya, covering demographic and mental health indicators.
- We loaded the data with the `mlcroissant` library and identified record sets and fields using their `@id`.
- Exploratory analysis demonstrated the distribution and relationships of key variables such as GAD-7, PHQ-9, or PSQ scores, grouped by demographic attributes like gender.
- This approach enables reproducible, standards-based exploration and supports further data science and machine learning workflows using Croissant data standards.